In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings("ignore")

In [8]:
ort = pd.read_excel("online_retail_cleaned.xlsx")

In [9]:
# ENSURE REQUIRED COLUMNS 
if "TotalPrice" not in ort.columns:
    ort["TotalPrice"] = ort["Quantity"] * ort["UnitPrice"]

ort["YearMonth"] = ort["InvoiceDate"].dt.to_period("M")
ort["Year"]      = ort["InvoiceDate"].dt.year

In [11]:
#  KPI SUMMARY 
total_revenue    = ort["TotalPrice"].sum()
total_orders     = ort["InvoiceNo"].nunique()
unique_customers = ort["CustomerID"].nunique()
avg_order_value  = total_revenue / total_orders

print(f"\n{'='*45}")
print(f"  RETAIL ANALYTICS SUMMARY")
print(f"{'='*45}")
print(f"  Total Revenue    : £{total_revenue:>12,.2f}")
print(f"  Total Orders     : {total_orders:>12,}")
print(f"  Unique Customers : {unique_customers:>12,}")
print(f"  Avg Order Value  : £{avg_order_value:>12,.2f}")
print(f"{'='*45}\n")


  RETAIL ANALYTICS SUMMARY
  Total Revenue    : £10,637,369.86
  Total Orders     :       19,947
  Unique Customers :        4,334
  Avg Order Value  : £      533.28



In [13]:
# REVENUE TRENDS OVER TIME
monthly_rev = (
    ort.groupby("YearMonth")["TotalPrice"]
    .sum()
    .reset_index()
    .rename(columns={"TotalPrice": "Revenue"})
)
monthly_rev["YearMonthStr"] = monthly_rev["YearMonth"].astype(str)

In [14]:
# Year-over-year comparison
yoy = ort.groupby("Year")["TotalPrice"].sum().rename("Revenue")

In [17]:
#  TOP SELLING PRODUCTS
top_products_rev = (
    ort.groupby("Description")["TotalPrice"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={"TotalPrice": "Revenue"})
)

top_products_qty = (
    ort.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

In [18]:
#  HIGH-VALUE CATEGORIES & REGIONS
# Revenue by country
country_rev = (
    ort.groupby("Country")
    .agg(
        Revenue=("TotalPrice", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Customers=("CustomerID", "nunique"),
    )
    .sort_values("Revenue", ascending=False)
    .reset_index()
)
country_rev["AOV"] = country_rev["Revenue"] / country_rev["Orders"]
country_rev["RevenueShare"] = country_rev["Revenue"] / country_rev["Revenue"].sum() * 100


In [19]:
# International (ex-UK) for chart clarity
intl_rev = country_rev[country_rev["Country"] != "United Kingdom"].head(8)

# Revenue by hour-of-day (purchasing pattern)
hourly_rev = ort.groupby("Hour")["TotalPrice"].sum()

In [20]:
#  COHORT: RETURNING vs NEW CUSTOMERS
customer_first = ort.groupby("CustomerID")["InvoiceDate"].min().dt.to_period("M")
customer_first.name = "FirstMonth"
ort_cohort = ort.join(customer_first, on="CustomerID")
ort_cohort["IsNew"] = ort_cohort["YearMonth"] == ort_cohort["FirstMonth"]
returning_pct = (~ort_cohort.groupby("InvoiceNo")["IsNew"].first()).mean() * 100

In [21]:
#  EXPORT SUMMARY TABLES
with pd.ExcelWriter("retail_analysis_output.xlsx", engine="openpyxl") as writer:
    # KPI sheet
    kpi_df = pd.DataFrame({
        "Metric":  ["Total Revenue (£)", "Total Orders", "Unique Customers", "Avg Order Value (£)", "Returning Customer %"],
        "Value":   [round(total_revenue, 2), total_orders, unique_customers, round(avg_order_value, 2), round(returning_pct, 1)],
    })
    kpi_df.to_excel(writer, sheet_name="KPIs", index=False)

    monthly_rev[["YearMonthStr", "Revenue"]].rename(
        columns={"YearMonthStr": "Month"}
    ).to_excel(writer, sheet_name="Monthly Revenue", index=False)

    top_products_rev.to_excel(writer, sheet_name="Top Products", index=False)
    country_rev.to_excel(writer, sheet_name="Country Revenue", index=False)




Saved - retail_analysis_output.xlsx
